# 🗄️ Day 154 — Streamlit + Database Integration
## Month 8 | Streamlit + FastAPI Deployment | Deepanshu Garg

---

| Field | Details |
|---|---|
| **Month** | 8 — Streamlit + FastAPI Deployment |
| **Day** | 154 (Month 8, Week 4, Day 4) |
| **Topic** | SQLite + SQLAlchemy inside Streamlit · `st.cache_resource` vs `st.cache_data` · parameterized CRUD · CSV bulk import · audit logging |
| **Dataset** | FreelanceHub India (500 rows, seed=141) |
| **Deliverables** | `day154_app.py` (standalone Streamlit app) · `Day154_Bulk_Import_Test.csv` |
| **Max Score** | 80 pts + 10★ Bonus |
| **GitHub Repo** | Month8-Streamlit-FastAPI-Portfolio |

---

## 📊 Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ ✅ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ ✅ |
| 143 | File Upload + Dynamic EDA | 80/80 + 10★ ✅ |
| 144 | ML Model Deployment (Basic) | 80/80 + 10★ ✅ |
| 145 | Advanced Caching + Multi-Page Apps | 80/80 + 10★ ✅ |
| 146 | ML Explainability Dashboard (SHAP) | 80/80 + 10★ ✅ |
| 147 | FastAPI Fundamentals | 80/80 + 10★ ✅ |
| 148 | FastAPI + DB + Auth | 80/80 + 10★ ✅ |
| 149 | Streamlit ↔ FastAPI Full-Stack | 80/80 + 10★ ✅ |
| 150 | Docker + Containerization | 80/80 + 10★ ✅ |
| 151 | Cloud Deployment (SCC + Render) | 80/80 + 10★ ✅ |
| 152 | CI/CD — GitHub Actions | 80/80 + 10★ ✅ |
| 153 | Monitoring & Structured Logging | 80/80 + 10★ ✅ |
| **154** | **Streamlit + Database Integration** | **← Today** |

**Running Total entering today: 1040/1040 + 130★ 🔥 PERFECT — 13 consecutive days**

---

## 🎯 Why This Day Matters

Day 148 gave the **FastAPI** layer a database (clients call an API, the API talks to SQLite).
Today gives the **Streamlit** layer its *own* direct database connection — no API in between.

```
Day 148 pattern                    Day 154 pattern
─────────────────────              ─────────────────────────────────
Browser → Streamlit                Browser → Streamlit
            ↓ HTTP                              ↓ direct SQLAlchemy call
          FastAPI                              SQLite (freelancehub.db)
            ↓ SQLAlchemy
          SQLite
```

> **Freelance reality:** Not every client needs a separate API. A solo internal tool —
> "let our ops team add/edit/delete projects without touching Excel" — is *faster and
> cheaper to build* as a single Streamlit app talking straight to a database. Knowing
> **when to skip the API layer** is itself a billable judgment call.

This is also the day caching gets dangerous for the first time: `st.cache_data` on a
function that reads the database will **silently show stale data** after an INSERT/UPDATE/
DELETE unless you explicitly invalidate it. That bug has burned real freelancers in
production — today you learn the fix before it costs you a client.


---
## 📂 Section 1 — Raw Data
> ⚠️ **NEVER MODIFY THIS SECTION.** `seed=141`, 500 rows, 11 columns.
> Same FreelanceHub India dataset used across all of Month 8 — today it becomes the
> seed data for a real SQLite database instead of just an in-memory DataFrame.


In [1]:
# ─── RAW DATA GENERATOR ─────────────────────────────────────────────────
# Purpose : Reproduce the Month 8 FreelanceHub India dataset (seed=141, 500 rows)
# Method  : Seeded random generation — IDENTICAL across every Month 8 day
#           so today's database is built from the exact same source of truth.

import pandas as pd
import numpy as np

np.random.seed(141)
n = 500

categories = ['ML/AI', 'Web Dev', 'Data Analytics', 'Graphic Design',
              'Content Writing', 'SEO/Digital Marketing']
platforms  = ['Upwork', 'Freelancer', 'Fiverr', 'Toptal', 'Direct Client']
cities     = ['Mumbai', 'Bangalore', 'Delhi', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
levels     = ['Entry', 'Mid', 'Senior']
outcomes   = ['Completed', 'Cancelled', 'In Progress']

df = pd.DataFrame({
    'project_id':    [f'PJ{1000+i}' for i in range(n)],
    'category':      np.random.choice(categories, n),
    'platform':      np.random.choice(platforms, n, p=[0.30, 0.25, 0.25, 0.10, 0.10]),
    'city':          np.random.choice(cities, n),
    'level':         np.random.choice(levels, n, p=[0.40, 0.40, 0.20]),
    'budget_inr':    np.random.randint(5000, 120001, n),
    'duration_days': np.random.randint(7, 91, n),
    'client_rating': np.round(np.random.uniform(2.5, 5.0, n), 1),
    'bids_received': np.random.randint(2, 51, n),
    'outcome':       np.random.choice(outcomes, n, p=[0.65, 0.20, 0.15]),
    'month_posted':  np.random.choice(['Jan','Feb','Mar','Apr','May','Jun',
                                       'Jul','Aug','Sep','Oct','Nov','Dec'], n),
})

# Introduce realistic nulls (same 25 rows every Month 8 day)
null_idx = np.random.choice(n, 25, replace=False)
df.loc[null_idx[:10],  'client_rating'] = np.nan
df.loc[null_idx[10:20],'bids_received'] = np.nan
df.loc[null_idx[20:],  'duration_days'] = np.nan

print(f"Shape: {df.shape}")
print(df.dtypes)
print()
print(df.head(5).to_string())


Shape: (500, 11)
project_id           str
category             str
platform             str
city                 str
level                str
budget_inr         int64
duration_days    float64
client_rating    float64
bids_received    float64
outcome              str
month_posted         str
dtype: object

  project_id         category       platform     city  level  budget_inr  duration_days  client_rating  bids_received    outcome month_posted
0     PJ1000          Web Dev         Fiverr     Pune  Entry      112435           17.0            2.8           29.0  Completed          Aug
1     PJ1001          Web Dev         Upwork  Chennai    Mid       95517           13.0            4.2           15.0  Completed          May
2     PJ1002   Graphic Design  Direct Client     Pune    Mid       71076           62.0            4.8           19.0  Cancelled          Jan
3     PJ1003   Data Analytics     Freelancer     Pune  Entry       53950           63.0            3.0           17.0  Comple

---
## 📚 Section 2 — Concept Notes

### 2.1 — Why SQLite (and not just `df.to_csv`)?

A DataFrame in memory dies the moment the Streamlit script reruns or the process restarts.
SQLite gives the app **persistence**: one file (`freelancehub.db`) survives restarts,
supports concurrent reads, and lets you query with SQL instead of re-filtering a DataFrame
every single rerun. For a solo internal tool, SQLite is the entire backend — no server to manage.

### 2.2 — Two different cache decorators, two different jobs

| Decorator | Caches | Use for | Cleared by |
|---|---|---|---|
| `st.cache_resource` | **Objects** (connections, models, engines) | One SQLAlchemy `Engine`, reused across reruns | Manual `.clear()` or app restart |
| `st.cache_data` | **Data** (DataFrames, JSON, query results) | Output of a `SELECT` query | TTL expiry, manual `.clear()`, or changed args |

**The bug this day exists to prevent:** if you wrap a `SELECT * FROM projects` call in
`st.cache_data` and then run an `INSERT`, the cached `SELECT` result is still the *old*
data. The user clicks "Add Project," sees a success message, then the table below still
shows the old row count — because Streamlit served the cache instead of re-querying.
**Fix:** call `load_projects.clear()` (or `st.cache_data.clear()`) immediately after any
write operation, before the rerun re-renders the table.

### 2.3 — Parameterized queries (the #1 security habit in this lesson)

```python
# ❌ NEVER — string-formatted SQL is a SQL-injection vector
query = f"SELECT * FROM projects WHERE platform = '{user_input}'"

# ✅ ALWAYS — bound parameters, the database library escapes the value safely
query = text("SELECT * FROM projects WHERE platform = :plat")
conn.execute(query, {"plat": user_input})
```

If `user_input` were `Upwork'; DROP TABLE projects; --`, the f-string version executes
that DROP. The parameterized version treats it as a harmless literal string. **Every
client-facing form input that reaches SQL must go through bound parameters — no exceptions.**

### 2.4 — CRUD inside Streamlit's rerun model

Streamlit reruns the *entire script* top-to-bottom on every interaction. CRUD writes
must therefore live inside `if st.button(...)` or `if st.form_submit_button(...)` blocks —
never at module level — or every single rerun (including unrelated ones) would re-execute
the INSERT/UPDATE/DELETE.

### 2.5 — Audit logging (today's bonus pattern)

A second table, `audit_log(timestamp, action, project_id, detail)`, written to on every
CRUD operation, turns "trust me, I updated it" into "here's the exact row, the exact
second, and who touched it." This is the single most-asked-for feature once a client's
internal tool has more than one user.


---
## 🛠️ Section 3 — Practice Guide

**Build everything below in `day154_app.py` in VS Code** (per the Month 8 workflow rule —
Streamlit apps are written directly as `.py`, never exported from a notebook). Use **this
notebook** only to design and verify your SQL/logic before transferring it into the app —
exactly like Day 148's `database.py` was prototyped before going into the FastAPI app.

Run `streamlit run day154_app.py` to test live. Submit `day154_app.py` for scoring.

| Task | Points |
|---|---|
| T1 — Database setup (df → SQLite) | 10 |
| T2 — Cache decorators (resource vs data) + cache invalidation | 15 |
| T3 — CRUD: parameterized INSERT / UPDATE / DELETE | 15 |
| T4 — CSV bulk import page | 15 |
| T5 — Parameterized filter/search function | 10 |
| T6 — NRA dashboard insight from a live query | 15 |
| ⭐ Bonus — Audit log table + "Recent Activity" sidebar feed | 10★ |
| **Total** | **80 + 10★** |

---

### Task 1 (10 pts) — Database setup

Using SQLAlchemy, create an engine pointing at `freelancehub.db` and write the Section 1
DataFrame into a table called `projects` **only if the table doesn't already exist** (so
re-running the app doesn't duplicate rows). Verify by reading the row count back.


In [2]:
# TODO — Task 1: build the engine + write the table once
# Hints:
#   - from sqlalchemy import create_engine, inspect
#   - engine = create_engine("sqlite:///freelancehub.db")
#   - inspect(engine).has_table("projects")  -> True/False, decide whether to write
#   - df.to_sql("projects", engine, if_exists="replace", index=False)

# Your code here:




### Task 2 (15 pts) — Caching: `st.cache_resource` vs `st.cache_data`

Write **two** functions for the app:
1. `get_engine()` — decorated with `@st.cache_resource`, returns the SQLAlchemy engine (created once).
2. `load_projects()` — decorated with `@st.cache_data(ttl=60)`, runs `SELECT * FROM projects`
   and returns a DataFrame.

Then write a one-line rule in a markdown comment: **after any INSERT/UPDATE/DELETE, you
must call `load_projects.clear()` before the next read** — otherwise the UI shows stale data.


In [3]:
# TODO — Task 2: get_engine() with st.cache_resource, load_projects() with st.cache_data
# (st is not available in this notebook context — write the logic with plain functions here,
#  then add the @st.cache_resource / @st.cache_data decorators when you paste into day154_app.py)

# Your code here:




### Task 3 (15 pts) — CRUD with parameterized queries

Implement three functions, **all using bound parameters** (never f-strings):

- `insert_project(engine, row: dict)` — INSERT one new row into `projects`.
- `update_outcome(engine, project_id: str, new_outcome: str)` — UPDATE the `outcome` column
  for one `project_id`.
- `delete_project(engine, project_id: str)` — DELETE one row by `project_id`.

**Test data for verification (use these exact values):**
- INSERT: `project_id='PJ2000', category='Data Analytics', platform='Upwork', city='Bangalore', level='Mid', budget_inr=75000, duration_days=30, client_rating=4.8, bids_received=12, outcome='In Progress', month_posted='Jun'`
- UPDATE: `project_id='PJ1100'` → set `outcome='Cancelled'` (this record's `bids_received` is currently NULL — a client audit flagged it as actually cancelled, not completed)
- DELETE: `project_id='PJ1250'` (confirmed duplicate test entry)


In [4]:
# TODO — Task 3: insert_project / update_outcome / delete_project
# Use sqlalchemy.text() with named parameters, e.g.:
#   conn.execute(text("DELETE FROM projects WHERE project_id = :pid"), {"pid": project_id})
# Remember: each write needs conn.commit() (or use engine.begin() as a context manager).

# Your code here:




### Task 4 (15 pts) — CSV bulk import page

Build a Streamlit page where the user uploads a CSV (matching the `projects` schema —
see `Day154_Bulk_Import_Test.csv` provided alongside this notebook). Validate that all 11
expected columns are present, then insert all rows into `projects` inside a single
transaction. Show a success message with the number of rows imported, and call
`load_projects.clear()` afterward.


In [5]:
# TODO — Task 4: read uploaded CSV, validate columns, bulk insert
# Hints:
#   - uploaded = st.file_uploader("Upload CSV", type="csv")
#   - new_df = pd.read_csv(uploaded)
#   - assert set(new_df.columns) == set(EXPECTED_COLUMNS), "Column mismatch!"
#   - new_df.to_sql("projects", engine, if_exists="append", index=False)

# Your code here (test against /mnt/user-data/outputs/Day154_Bulk_Import_Test.csv style data):




### Task 5 (10 pts) — Parameterized filter/search function

Write `filter_projects(engine, category=None, platform=None, outcome=None)` that builds a
parameterized `WHERE` clause dynamically (only filtering on the arguments that aren't
`None`) and returns the matching rows as a DataFrame.

**Verification case:** `filter_projects(engine, category='ML/AI', platform='Upwork')`
should return **28 rows** with an average `budget_inr` of **₹57,816.79**.


In [6]:
# TODO — Task 5: dynamic parameterized WHERE clause
# Hints:
#   - build a list of "column = :paramname" clauses only for non-None args
#   - join with " AND ", default to "1=1" if no filters given
#   - pass a dict of only the active params to conn.execute

# Your code here:




### Task 6 (15 pts) — NRA dashboard insight from a live query

Query the **current** `projects` table (after Task 3's UPDATE) and compute the
cancellation rate (% of rows where `outcome='Cancelled'`) **grouped by platform**. Write
one NRA (Number + Reason + Action) insight about the platform with the highest
cancellation rate.


In [7]:
# TODO — Task 6: cancellation rate % by platform from a live SELECT
# Hint: pd.read_sql("SELECT platform, outcome FROM projects", engine), then groupby + mean

# Your code here:


# Write your NRA insight as a comment below:
# Number  :
# Reason  :
# Action  :


### ⭐ Bonus (10★) — Audit log + Recent Activity feed

Create a second table `audit_log(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT,
action TEXT, project_id TEXT, detail TEXT)`. Every INSERT/UPDATE/DELETE/bulk-import from
Tasks 3–4 should also write one row here (`ts` = `datetime.now().isoformat()`). Add a
Streamlit sidebar section showing the **5 most recent** audit log entries.


In [8]:
# TODO — Bonus: create audit_log table + log_action() helper + recent-activity query

# Your code here:




---
## 🧮 Section 4 — Scoring Rubric

| # | Task | Points | What earns full marks |
|---|---|---|---|
| T1 | Database setup | 10 | Engine created, table written once, `inspect().has_table()` guard present, row count = 500 |
| T2 | Caching decorators | 15 | `@st.cache_resource` on engine, `@st.cache_data(ttl=...)` on query, explicit `.clear()` call after every write |
| T3 | CRUD operations | 15 | All 3 functions use `text()` + bound params (zero f-string SQL), correct INSERT/UPDATE/DELETE results match verification rows |
| T4 | CSV bulk import | 15 | Column validation present, transactional append, success message with row count, cache cleared after |
| T5 | Parameterized filter | 10 | Dynamic WHERE clause from non-None args only, bound params, returns 28 rows / ₹57,816.79 avg for the test case |
| T6 | NRA insight | 15 | Correct live cancellation-rate numbers (Toptal 23.73%, overall 20.4% — measured live after Task 3 AND Task 4), full Number+Reason+Action, Action is specific and committal |
| ⭐ Bonus | Audit log + Recent Activity | 10★ | Table auto-created, every write logged, sidebar shows last 5 entries newest-first |

**Communication vs Technical split (tracked separately per the program rules):**
- **Technical** failure example: using f-string SQL anywhere → automatic -5 on that task (security habit, non-negotiable).
- **Communication** failure example: correct cancellation numbers but Action says "investigate Toptal further" instead of naming a specific threshold/process → -2 on T6 even if the Number/Reason are perfect.


---
## 🔑 Key Takeaway — Day 154

> **A cache that doesn't know about your writes is worse than no cache at all** — it doesn't
> just fail to help, it actively lies to the screen. The professional habit isn't
> "remember to clear the cache" (you'll forget under deadline pressure); it's **always pair
> a write function with its cache-clear call in the same code block**, so the two can never
> be separated by accident. Every Streamlit-plus-database bug you'll hit in freelance work
> traces back to this one pairing being broken somewhere.


---
## 🎤 Interview Angle

**Q: "Walk me through how you'd add a database to a Streamlit app."**

> "I use SQLAlchemy with a cached engine — `st.cache_resource` so the connection isn't
> rebuilt on every rerun — and I separate that from `st.cache_data` on the actual query
> functions, with an explicit `.clear()` right after any write. All writes go through
> parameterized queries with `text()` and bound params, never string formatting, because
> Streamlit forms are direct user input and that's a SQL-injection surface the moment you
> skip it. For anything beyond a single-user internal tool, I'd put a FastAPI layer in
> front of the same database — which is exactly what I built in Month 8, Day 148 — but for
> a fast internal ops tool, going Streamlit-straight-to-SQLite is often the right call: less
> infrastructure, same data integrity guarantees, shipped in a day instead of a week."

This answer demonstrates: caching internals (not just "I used cache"), a named security
habit, and — critically — judgment about *when not* to over-engineer with an API layer.


---
### The Day 154 Stack (your deliverable)

```
        ┌─────────────────────────────┐
        │   day154_app.py (Streamlit) │
        │                              │
        │  Page 1: Dashboard           │ ← T6 NRA + live cancellation chart
        │  Page 2: Add / Edit / Delete │ ← T3 CRUD (parameterized)
        │  Page 3: Bulk CSV Import     │ ← T4
        │  Page 4: Filter & Search     │ ← T5
        │  Sidebar: Recent Activity    │ ← Bonus audit log
        └───────────────┬──────────────┘
                         │ st.cache_resource (engine)
                         │ st.cache_data + .clear() (queries)
                         ▼
              ┌─────────────────────┐
              │  freelancehub.db     │
              │  ├── projects        │
              │  └── audit_log       │
              └─────────────────────┘
```

Every box above maps to a graded task. Build it in that order — database, then caching,
then CRUD, then import/filter, then the audit trail — and `streamlit run day154_app.py`
should work end-to-end on the first try.

**GitHub:** commit `day154_app.py`, `Day154_Bulk_Import_Test.csv`, and this notebook to
`Month8-Streamlit-FastAPI-Portfolio` — `auto_sync.py` will pick them up automatically.
